In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")

In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================

FILE_PATH = "regression_data.csv"

df = pd.read_csv(FILE_PATH)

print(df.head())
print(df.shape)
df.info()
print(df.describe(include="all").T)

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

TARGET = "target"
ID_COLUMNS = []

CATEGORICAL_COLUMNS = []
NUMERIC_COLUMNS = []

feature_columns = [
    c for c in df.columns
    if c not in [TARGET] + ID_COLUMNS
]

if not CATEGORICAL_COLUMNS:
    CATEGORICAL_COLUMNS = (
        df[feature_columns]
        .select_dtypes(include=["object", "category", "bool"])
        .columns.tolist()
    )

if not NUMERIC_COLUMNS:
    NUMERIC_COLUMNS = [
        c for c in feature_columns
        if c not in CATEGORICAL_COLUMNS
    ]

print("Categorical:", CATEGORICAL_COLUMNS)
print("Numeric:", NUMERIC_COLUMNS)

In [ ]:
# ============================================================
# 3. EDA
# ============================================================

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicates:", df.duplicated().sum())

plt.figure(figsize=(7, 4))
sns.histplot(df[TARGET], kde=True)
plt.title("Target Distribution")
plt.tight_layout()
plt.show()

if NUMERIC_COLUMNS:
    corr_columns = NUMERIC_COLUMNS + [TARGET]
    corr = df[corr_columns].corr()

    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()

    target_corr = (
        corr[TARGET]
        .drop(TARGET)
        .sort_values()
    )

    target_corr.plot(kind="barh", figsize=(8, 6))
    plt.title("Correlation with Target")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 4. FEATURES / TARGET / SPLIT
# ============================================================

X = df.drop(columns=[TARGET] + ID_COLUMNS)
y = df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
)

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
# ============================================================
# 5. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_COLUMNS),
    ("cat", categorical_pipeline, CATEGORICAL_COLUMNS),
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
# ============================================================
# 6. MODELS
# ============================================================

linear_model = LinearRegression()

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1,
)

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

models = {
    "Linear Regression": linear_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    print(name, "trained")

In [ ]:
# ============================================================
# 7. EVALUATION
# ============================================================

def evaluate_model(model, X, y, model_name):
    predictions = model.predict(X)

    mae = mean_absolute_error(y, predictions)
    rmse = np.sqrt(mean_squared_error(y, predictions))
    r2 = r2_score(y, predictions)

    print(f"\n========== {model_name} ==========")
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2  : {r2:.4f}")

    return {
        "model": model_name,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
    }

results = []

for name, model in models.items():
    results.append(
        evaluate_model(model, X_val_processed, y_val, name)
    )

results_df = pd.DataFrame(results).sort_values(
    "rmse"
)

print(results_df)

In [ ]:
# ============================================================
# 8. RESIDUAL PLOT
# ============================================================

BEST_MODEL_NAME = results_df.iloc[0]["model"]
best_model = models[BEST_MODEL_NAME]

val_predictions = best_model.predict(X_val_processed)
residuals = y_val - val_predictions

plt.figure(figsize=(7, 4))
sns.scatterplot(x=val_predictions, y=residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title(f"Residual Plot - {BEST_MODEL_NAME}")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9. FINAL TEST
# ============================================================

test_predictions = best_model.predict(X_test_processed)

print("Selected model:", BEST_MODEL_NAME)
print("MAE :", mean_absolute_error(y_test, test_predictions))
print(
    "RMSE:",
    np.sqrt(mean_squared_error(y_test, test_predictions))
)
print("R2  :", r2_score(y_test, test_predictions))